---
title: "Informe de evaluación del proceso de trámite de Peticiones, Quejas y Reclamos"
format:
  html:
    page-layout: full
jupyter: python3
---

# 1. Cargar datos generados por el sistema de gestión documental (SGD)

**Cargar librerías**

In [2]:
import os
import numpy as np
import pandas as pd

## 1.1 Leer los archivos planos

In [3]:
# Crear un listado de todos los archivos que están en la carpeta datos
# En la carpeta datos están almacenados todos los archivos con la información indumo para el informe
# archivos_pqrs = os.listdir("proyectos/freelance_informe_PQRS/datos")
archivos_pqrs = os.listdir("datos")
# Generar un diccionario vacío para guardar los archivos pqrs que se generaron en el SGD
datos_pqrs = dict()

# Leer todos los archivos de la carpeta datos que son de tipo hoja de cálculo de Excel
# El SGD genera los archivos en formato Excel
for archivo in archivos_pqrs:
    if archivo.endswith((".xls", ".xlsx")):

        datos_pqrs[archivo] = pd.read_excel(f"datos/{archivo}",
            #f"proyectos/freelance_informe_PQRS/datos/{archivo}",
                                      keep_default_na=False,
                                      na_filter=False)

## 1.2 Crear los conjuntos de datos

In [4]:
# Crear dos dataframes: uno de radicados y el otro de respuestas
respuestas = pd.DataFrame()
radicados = pd.DataFrame()

# Concatenar los dataframes 
for key in datos_pqrs.keys():
    if "Respuestas" in key:
        respuestas = pd.concat([respuestas,datos_pqrs[key]], axis = 0, ignore_index=True)
    else:
        radicados = pd.concat([radicados,datos_pqrs[key]], axis = 0, ignore_index=True)

**Previsualización de los conjuntos de datos**

In [5]:
# Vista de los dataframes
print('Vista previa del conjunto de datos de respuestas a comunicaciones: ')
print(respuestas.sample(5))
print()
print('Vista previa del conjunto de datos de comunicaciones radicadas: ')
print(radicados.sample(5))

Vista previa del conjunto de datos de respuestas a comunicaciones: 
       NroRadicado  ...                                          Respuesta
13084   2019121110  ...  2019602934 (2019-08-27 16:13:30.0), 2019602935...
17273   2019125358  ...                 2019598006 (2019-08-15 15:18:14.0)
108643  2019093452  ...                 2019593221 (2019-08-06 18:29:51.0)
59644   2019063780  ...                 2019547156 (2019-05-02 16:39:33.0)
57663   2019061453  ...                 2019540479 (2019-04-15 10:52:04.0)

[5 rows x 5 columns]

Vista previa del conjunto de datos de comunicaciones radicadas: 
       NroRadicado  ...                       Descripcion
36659   2019558744  ...  respuesta al radicado 2019098465
41862   2019563948  ...  respuesta al radicado 2019110550
51701   2019521330  ...  respuesta al radicado 2019027710
9350    2019517926  ...  respuesta al radicado 2019029602
1332    2019509902  ...  respuesta al radicado 2019009174

[5 rows x 5 columns]


**Características de los conjuntos de datos**

In [6]:
# Detalles de los tipos de datos y cantidades no nulas
print("Detalles del dataframe de radicados de respuestas a documentos recibidos: \n")
print(respuestas.info())
print()
print("Detalles del dataframe de comunicaciones radicadas: \n")
print(radicados.info())

Detalles del dataframe de radicados de respuestas a documentos recibidos: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119949 entries, 0 to 119948
Data columns (total 5 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   NroRadicado      119949 non-null  object
 1   Asunto           119949 non-null  object
 2   FechaRadicacion  119949 non-null  object
 3   Ruta             119949 non-null  object
 4   Respuesta        119949 non-null  object
dtypes: object(5)
memory usage: 4.6+ MB
None

Detalles del dataframe de comunicaciones radicadas: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72616 entries, 0 to 72615
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   NroRadicado      72616 non-null  int64 
 1   Asunto           72616 non-null  object
 2   FechaRadicacion  72616 non-null  object
 3   TipoDocumental   72616 non-null  object
 4   Descripci

# 2. Pre-procesamiento de datos: limpieza y transformación de los conjuntos de datos

## 2.1 Modificación del tipo de variables

Se cuenta con dos *datasets*: el primero agrupa la información de las respuestas a comunicaciones recibidas y el segundo corresponde a las comunicaciones radicadas. Ambos contienen cinco variables, entre las cuales se encuentra **FechaRadicacion**, que registra la fecha de radicación de un documento en el Sistema de Gestión Documental (SGD). En ambos casos este campo se encuentra definido inicialmente como tipo *object*, es decir, los valores se interpretan como texto y no como fechas. Dado que posteriormente será necesario realizar análisis por periodos de tiempo, resulta indispensable ajustar el tipo de dato a formato de fecha.

Adicionalmente, la variable **NroRadicado**, común a ambos conjuntos de datos, presenta diferencias en su tipificación: en el dataframe *respuestas* se encuentra como tipo *object*, mientras que en el dataframe *radicados* aparece como tipo entero. Considerando que más adelante se requerirá combinar ambos conjuntos a través de esta variable, es pertinente unificar su tipo de dato para garantizar la correcta integración y consistencia en el procesamiento.

In [7]:
# Cambiar el tipo de dato "object" a "fecha" (formato: YYYY-MM-DD HH:MM:SS.0) en los campos "FechaRadicacion"

# En el dataframe de respuetas
respuestas["FechaRadicacion"] = pd.to_datetime(
    respuestas["FechaRadicacion"],
    format="%Y-%m-%d %H:%M:%S.%f",
    errors="coerce"
)

# En el dataframe de radicados
radicados["FechaRadicacion"] = pd.to_datetime(
    radicados["FechaRadicacion"],
    format="%d/%m/%Y %H:%M:%S.%f",
    errors="coerce"
)

In [8]:
# Cambiar el tipo de dato "object" a "entero" en el campo "NroRadicado" en el dataframe "respuestas"

respuestas["NroRadicado"] = respuestas["NroRadicado"].astype('int')

## 2.2 Generación de variables adicionales

La variable **Respuesta** almacena (del dataframe "respuestas"), en un único campo, la información de una o varias respuestas asociadas a la misma comunicación radicada. Cada respuesta se registra con el número de radicado y la fecha correspondiente, separados por comas cuando existen múltiples valores. Esta estructura implica que un mismo registro puede contener desde una hasta varias respuestas (en algunos casos más de cuarenta), lo que dificulta su análisis directo.

Para disponer de la información de manera estructurada y facilitar su procesamiento, es necesario **descomponer el contenido de la variable en columnas independientes**, de modo que cada respuesta quede representada en una variable distinta. En los casos en que un registro tenga menos respuestas que el máximo identificado, las columnas adicionales se completarán con valores nulos (*NaN*). Esta transformación garantiza que cada respuesta pueda ser tratada como un dato individual, permitiendo análisis más precisos y consistentes.

In [9]:
# Dividir por comas (ejemplo de la info actual: 2019557382 (2019-05-27 08:35:31.0), 2019557381...)
# Creando un dataframe auxiliar con las nuevas columnas
respuestas_division = respuestas["Respuesta"].str.split(",", expand=True)

# Renombrar de manera dinámica las columnas del dataframe auxiliar
respuestas_division.columns = [f"Respuesta_{i+1}" for i in range(respuestas_division.shape[1])]

# Agregar las columnas del dataframe auxiliar al dataframe de respuestas
respuestas = respuestas.join(respuestas_division)

In [10]:
print(f"la mayor cantidad de respuestas registradas para un solo radicado es de: {len(respuestas_division.columns)}")

la mayor cantidad de respuestas registradas para un solo radicado es de: 41


Adicional a la cantidad de respuestas por radicado es posible que el dataframe "respuesta" tenga más de un registro para el mismo radicado, esto se debe a que las respuestas se pueden haber tramitado por más de una "Ruta". El valor exacto de comunicaciones radicadas y respuestas generadas (esto es, registros únicos), según reporte del SGD, se observa así:

In [11]:
# Número de radicados

unicos_respuestas = respuestas['NroRadicado'].nunique()
unicos_radicados = radicados['NroRadicado'].nunique()

print(f"{unicos_radicados:,} comunicaciones recibidas o radicados y {unicos_respuestas:,} respuestas generadas")

72,615 comunicaciones recibidas o radicados y 50,735 respuestas generadas


Los datos disponibles en este análisis abarcan un periodo de seis meses

In [12]:
# Periodo de tiempo de datos disponibles

desde = min(min(respuestas['FechaRadicacion']), min(radicados['FechaRadicacion']))
hasta = max(max(respuestas['FechaRadicacion']), max(radicados['FechaRadicacion']))

print(f"Los registros incluyen datos desde el día {desde.strftime('%Y-%m-%d')} hasta el día  {hasta.strftime('%Y-%m-%d')}")

Los registros incluyen datos desde el día 2019-01-02 hasta el día  2019-06-30


In [13]:
radicados.head()

,NroRadicado,Asunto,FechaRadicacion,TipoDocumental,Descripcion
0,2019508566,por defecto,2019-02-01 07:21:21,comunicacion,respuesta al radicado 2019016057
1,2019508567,por defecto,2019-02-01 07:26:10,comunicacion,respuesta al radicado 2019016057
2,2019508568,por defecto,2019-02-01 07:51:46,comunicacion,accion popular universidad de estado
3,2019508569,por defecto,2019-02-01 07:55:07,comunicacion,notificacion de aviso
4,2019508570,por defecto,2019-02-01 07:57:18,comunicacion,union temporal cardiovascular del nino


# 3. Identificación del conjunto de datos relevante

El propósito de este informe es evaluar el trámite de las Peticiones, Quejas y Reclamos (PQRS), es decir, la gestión de las respuestas a las comunicaciones que recibe la entidad y que deben ser atendidas para los usuarios que las radican. Dado que no toda la información disponible en los conjuntos de datos corresponde a PQRS, resulta necesario identificar y seleccionar únicamente aquellas que sí lo son. Para lograrlo, se requiere filtrar los registros de acuerdo con la información específica contenida en determinados campos de interés de cada *dataframe*.

**Radicados**

-   **Campo *TipoDocumental***: se deben seleccionar los registros que incluyan las palabras *“PQRD”*, *“PETICION”* o *“QUEJA”*.

-   **Campo *Descripción***: se deben seleccionar los registros que incluyan los textos *“PETICION”*, *“PECTION”*, *“PETCIOM”*, *“EPTICION”*, *“PECTICION”*, *“PETICON”*, *“DERECHO”*, *“DERCHO”* y *“DERECH”*.

**Respuestas**

-   **Campo *Asunto***: se deben seleccionar los registros que incluyan los textos *“PETICION”*, *“PETICIONES”*, *“PQRD”*, *“PQRSD”* o *“QUEJA”*.

-   **Campo *Ruta***: se deben seleccionar los registros que incluyan el texto *“PQRS”*.

In [14]:
# Definir los criterior o textos de filtro para cada campo

# Criterios de filtro para el dataframe "radicados"
tipo = ["PQRD","PETICION","QUEJA"]
descripcion = ["PETICION", "PECTION", "PETCIOM", "EPTICION", "PECTICION", "PECTION", "PETICON", "DERECHO", "DERCHO", "DERECH"]

# Criterios de filtro para el dataframe "respuestas"
asunto = ["PETICION", "PETICIONES", "PQRD", "PQRSD", "QUEJA"]
ruta = "PQRS"

## 3.1 Consolidación de PQRS radicadas

In [15]:
# Identificar en qué registros se incluyen los criterios de filtro de texto (verdadero o falso) del dataframe "radicados"

tipo_ = radicados['TipoDocumental'].str.contains("|".join(tipo), case=False, na=False)
descripcion_ = radicados['Descripcion'].str.contains("|".join(descripcion), case=False, na=False)

In [16]:
# Extraer los índices de PQRS identificados en "radicados"

filtro_radicados = list()
i = 0
while i <= len(radicados) - 1:
    if tipo_[i] or descripcion_[i]:
        filtro_radicados.append(i)
    i += 1

In [17]:
# Crear el conjunto de datos de PQRS radicadas

pqrs_radicados = radicados.iloc[filtro_radicados,:]

In [18]:
print(f"Para el período en evaluación se cuenta con un total de {pqrs_radicados['NroRadicado'].nunique():,} documentos de los cuales no se cuenta con información suficiente para \nestablecer si los mismos de requerian de respuesta alguna.")

Para el período en evaluación se cuenta con un total de 584 documentos de los cuales no se cuenta con información suficiente para 
establecer si los mismos de requerian de respuesta alguna.


El reporte del Sistema de Gestión Documental registra un total de 72.615 comunicaciones recibidas o radicadas. Al aplicar el filtro de posibles PQRS, se identifican 584 radicados; sin embargo, es posible que dichos documentos no estén sujetos a respuesta, ya que la información disponible no permite concluir con certeza que requieran una contestación. En consecuencia, se anticipa que las conclusiones presentadas en este documento podrían contener cierto grado de imprecisión. No obstante, dicho margen sería mínimo, considerando que frente al total de 50.735 respuestas generadas, el impacto proporcional resulta reducido.

## 3.2 Consolidación de PQRS con respuesta

In [19]:
# Identificar en qué registros se incluyen los criterios de filtro de texto (verdadero o falso) del dataframe "respuestas"

asunto_ = respuestas['Asunto'].str.contains("|".join(asunto), case=False, na=False)
ruta_ = respuestas['Ruta'].str.contains("|".join(ruta), case=False, na=False)

In [20]:
# Extraer los índices de PQRS identificados en "respuestas"

filtro_respuestas = list()
i = 0
while i <= len(respuestas) - 1:
    if asunto_[i] or ruta_[i]:
        filtro_respuestas.append(i)
    i += 1

In [21]:
# Crear el conjunto de datos de respuestas a PQRS

pqrs_respuestas = respuestas.iloc[filtro_respuestas,:]

In [26]:
print(f"Para el período en evaluación se cuenta con un total de {pqrs_respuestas['NroRadicado'].nunique():,} documentos que requerían respuesta, de las cuales se tiene información \nde la contestación suministrada al interesado(a).")
print(f"La cantidad de respuestas generadas, según el sistema de gestión documental, asciende a {len(pqrs_respuestas['NroRadicado']):,}.")

Para el período en evaluación se cuenta con un total de 30,075 documentos que requerían respuesta, de las cuales se tiene información 
de la contestación suministrada al interesado(a).
La cantidad de respuestas generadas, según el sistema de gestión documental, asciende a 49,777.


Aunque inicialmente se menciona que solo 584 documentos son identificados como PQRS, el número de registros (que supera los 70.000) incluye documentos radicados que no se identifican como sujetos de respuesta al ser radicados pero que durante su trámite si se marcan como tal. La cantidad de documentos radicados tipo PQRS tramitados en el proceso fue de 30.659, adicionalmente, la cantidad de documentos generados como respuesta es de 49.777 (número prerliminar).

## 3.3 Conjunto de documentos radicados consolidados

In [23]:
# Excluir los radicados que ya están como respuestas

pqrs_radicados = pqrs_radicados[~pqrs_radicados['NroRadicado'].isin(pqrs_respuestas['NroRadicado'])]

In [24]:
# Unir los datas de respuestas y radicados en una sola, reemplazar NaN por None

pqrs_matriz = pd.concat([pqrs_radicados,pqrs_respuestas], axis = 0)
pqrs_matriz = pqrs_matriz.replace({np.nan: None})

In [ ]:
print(f"El total de documentos radicados identificados como sujetos de respuesta (PQRS) es de  {pqrs_matriz['NroRadicado'].nunique():,}.")

Para el período en evaluación se cuenta con un total de 30,659 documentos tipo PQRS que requieren 
respuesta, de los cuales 584 no se cuenta con información de respuesta alguna o si el documento requería respuesta.


La evaluación de la gestión de PQRS tendrá en cuenta 30.075 radicados, aquellos que tienen información de: áreas de negocio que participaron en la gestión de la respuesta, fechas de radicado y fechas de respuesta, entre otros datos. Adicional se cuenta con 49.777 registros de respuesta, el cual es un dato preliminar ya que puede haber duplicidad de registros o documentos de respuesta cancelados.

# 4. Análisis de la gestión de PQRS